In [ ]:
from torn.api import TornAPI
from torn.watermark import WatermarkManager
from databricks.storage import Storage

base_location = "torn/user/user_api_files"

tables = ["attacks"]

table_params = {
    "attacks": {"sort": "ASC"}
}

DEFAULT_FROM_TS = 1681484237

torn_api = TornAPI(key=dbutils.secrets.get('Personal', 'TornAPI'), category="user")

db_storage = Storage(spark)
wm = WatermarkManager(spark, DEFAULT_FROM_TS, category="user")

In [ ]:
for table in tables:
    params = dict(table_params[table])
    run, params = wm.get_run_params(table, params)

    if run:
        table_param = torn_api.prep_params(**params)
        fac_json = torn_api.get_json(table, table_param)
        if table in ["balance", "basic", "members"]:
            db_storage.store(fac_json, base_location + table, merge_schema=True, add_date=True)
        else:
            db_storage.store(fac_json, base_location + table, merge_schema=True)